## Import and load Datasets 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
dataset_paths = {'customers' : '../data/olist_customers_dataset.csv',
"geolocation" : "../data/olist_geolocation_dataset.csv",
"order_items" : "../data/olist_order_items_dataset.csv",
"order_payments" : '../data/olist_order_payments_dataset.csv',
"order_reviews" : '../data/olist_order_reviews_dataset.csv',
"orders" : '../data/olist_orders_dataset.csv',
"products" : '../data/olist_products_dataset.csv',
"sellers" : '../data/olist_sellers_dataset.csv',
"product_category" : '../data/product_category_name_translation.csv'}


In [ ]:
# Load dataset

def load_dataset():

    data = {}
    for key in dataset_paths:
         data[key]= pd.read_csv(dataset_paths[key])

    return data

data = load_dataset()

In [ ]:
df = data['orders'].merge(data['order_items'], on='order_id', how='inner')
df = df.merge(data['order_payments'], on='order_id', how='inner')
df = df.merge(data['order_reviews'], on='order_id', how='inner')
df = df.merge(data['products'], on='product_id', how='inner')
df = df.merge(data['customers'], on='customer_id', how='inner')
df = df.merge(data['sellers'], on='seller_id', how='inner')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df['order_purchase_timestamp']

In [ ]:
# Ensure timestamps are in datetime format
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])

# Create useful features from order_purchase_timestamp
df['day_of_week_int'] = df['order_purchase_timestamp'].dt.weekday + 1  # Day of week as integer (1 = Monday, etc.)
df['hour'] = df['order_purchase_timestamp'].dt.hour                    # Hour of day
df['month'] = df['order_purchase_timestamp'].dt.month                  # Month as integer
df['year'] = df['order_purchase_timestamp'].dt.year                    # Year as integer
df['date'] = df['order_purchase_timestamp'].dt.to_period('M')          # Monthly period for time series analysis

# Calculate delivery time in days
df['delivery_time'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

df.head()


In [ ]:
# Renaming the column to correct the spelling
df.rename(columns={'product_name_lenght': 'product_name_length'}, inplace=True)

## Exploratory Data Analysis

In [ ]:
import seaborn as sns
# Set color palette for Seaborn
colors = ["#26536f", "#3b96b7", "#749ca8", "#b6a98d", "#c78a4d", "#854927"]
sns.set_palette(colors)

In [ ]:
#monthly sales trend
monthly_sales = df.groupby(df['order_purchase_timestamp'].dt.to_period('M')).agg({'price':'sum'})

# set figure size 
plt.figure(figsize=(12,6))

# make plot
plt.plot(monthly_sales.index.astype(str), monthly_sales['price'], marker='o' ,linestyle='-' , color=colors[0], linewidth=2)

#titles, labels, ticks
plt.title('Monthly sales Trend', fontsize=16 , fontweight='bold')
plt.xlabel('Month',fontsize=14)
plt.ylabel('Total Sales', fontsize=14)

plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)

plt.grid(visible=True, linestyle='--', alpha=0.65)

plt.tight_layout()
plt.show()

In [ ]:
#sales by category

#plot
plt.figure(figsize=(12,9))

top_categories = df['product_category_name'].value_counts().head(10)
top_categories.plot(kind='bar', color=colors[1], edgecolor=colors[0])

#title labels ticks
plt.title('Top 10 Product Categories', fontsize=16, fontweight='bold')
plt.xlabel('Product Category', fontsize=14)
plt.ylabel('Number of orders', fontsize=14)

plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)

plt.grid(axis='y', linestyle='--', alpha=0.65)

plt.tight_layout()
plt.show()

In [ ]:
# proportion of total sales by payment type

#plot
plt.figure(figsize=(8,8))

sales_by_payment = df.groupby('payment_type')['price'].sum()
sales_by_payment.plot(kind='pie', autopct='%1.1f%%',startangle=90, colors=colors)

#title
plt.title('Sales Distribution by Payment Type', fontsize=16, fontweight='bold')

plt.ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
#distribution of review score

plt.figure(figsize=(10,6))

df['review_score'].hist(bins=5, color=colors[1], edgecolor='black')

#title
plt.title('Distribution of review Scores', fontsize=16, fontweight='bold')
plt.xlabel('Review Score', fontsize=14)
plt.ylabel('Frequency',fontsize=14)

plt.grid(axis='y',linestyle='--', alpha=0.65)

plt.tight_layout()
plt.show()

In [ ]:
#corelation

correlation_features= df[['price', 'review_score', 'delivery_time','payment_value', 'freight_value', 'payment_installments', 'order_item_id','hour','month']]

#correlation matrix
correlation = correlation_features.corr()

#colormap
custom_colors = colors
custom_cmap = LinearSegmentedColormap.from_list('custom_cmap', custom_colors)

plt.figure(figsize=(10,8))
sns.heatmap(correlation, annot=True, cmap=custom_cmap, vmin=-1, vmax=1, fmt='.2f', linewidths=.5)
plt.title('correlation heatmap')
plt.show()

In [ ]:
# delivery time by product category
category_counts = df['product_category_name'].value_counts()

#threshold to decide which categories to keep
threshold = 3000
common_categories = category_counts[category_counts >= threshold].index

# Create a new column for simplified categories
df['simplified_category'] = df['product_category_name'].where(df['product_category_name'].isin(common_categories),'other')

In [ ]:
#box plot
plt.figure(figsize=(12,8))

sns.boxplot(x='delivery_time', y='simplified_category', data=df, palette=colors)

# Overlay swarm plot to show individual points
# sns.swarmplot(x='delivery_time', y='simplified_category', data=df, color='k', alpha=0.6, size=3)

#titles
plt.title('Delivery Time by Product Category (simplified)', fontsize=16, fontweight ='bold')
plt.xlabel('Delivery Time (days)', fontsize=14)
plt.ylabel('Product Category', fontsize=14)

plt.grid(axis='x', linestyle='--', alpha=0.65)

plt.tight_layout()
plt.show()

In [ ]:
#delivery time by day of the week

plt.figure(figsize=(12,6))
sns.violinplot(data=df, x='day_of_week_int', y='delivery_time', palette=colors)

#title
plt.title('Delivery time by Day of the Week', fontsize=16)
plt.xlabel('Day of the Week', fontsize=14)
plt.xticks(ticks=range(7), labels=['Mon', 'Tue', 'Wed', 'Thu','Fri','Sat','Sun'])
plt.grid(axis='y', linestyle='--', alpha=0.65)

plt.tight_layout()
plt.show()

In [ ]:
#average delivery time by month

df['month_year'] = df['order_purchase_timestamp'].dt.to_period('M')
monthly_delivery_time = df.groupby('month_year')['delivery_time'].mean().reset_index()

#plot
plt.figure(figsize=(12,6))
sns.barplot(x='month_year', y='delivery_time', data=monthly_delivery_time, palette=colors)

#title
plt.title('Average Delivery Time by Month', fontsize=16, fontweight='bold')
plt.xlabel('Month-year', fontsize=14)
plt.ylabel('Average Delivery Time (Days)', fontsize=14)

plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.65)
plt.tight_layout()
plt.show()